In [25]:
import pandas as pd
import numpy as np
from pandas.testing import assert_frame_equal

In [26]:
# ----------------------------------------------------------------------
# Solution function
def solution(library_books: pd.DataFrame, borrowing_records: pd.DataFrame) -> pd.DataFrame:
    """
    Returns books with zero available copies (total_copies - current_borrowers == 0).
    Ordered by current_borrowers DESC, title ASC.
    """
    # Count current borrows per book (return_date is NULL)
    current_borrows = (
        borrowing_records[borrowing_records['return_date'].isna()]
        .groupby('book_id')
        .size()
        .reset_index(name='current_borrowers')
    )
    
    # Merge with library_books
    merged = library_books.merge(current_borrows, on='book_id', how='left')
    merged['current_borrowers'] = merged['current_borrowers'].fillna(0).astype(int)
    
    # Filter books with zero available copies
    result = merged[merged['total_copies'] - merged['current_borrowers'] == 0]
    
    # Sort
    result = result.sort_values(
        by=['current_borrowers', 'title'],
        ascending=[False, True]
    )
    
    # Select columns as per example
    return result[['book_id', 'title', 'author', 'genre', 'publication_year', 'current_borrowers']]

# ----------------------------------------------------------------------
# Helper to compute expected result (independent of solution)
def compute_expected(library_books_df: pd.DataFrame, borrowing_records_df: pd.DataFrame) -> pd.DataFrame:
    # Count current borrows
    current_borrows = (
        borrowing_records_df[borrowing_records_df['return_date'].isna()]
        .groupby('book_id')
        .size()
        .reset_index(name='current_borrowers')
    )
    merged = library_books_df.merge(current_borrows, on='book_id', how='left')
    merged['current_borrowers'] = merged['current_borrowers'].fillna(0).astype(int)
    # Filter
    qualified = merged[merged['total_copies'] - merged['current_borrowers'] == 0]
    # Sort
    qualified = qualified.sort_values(
        by=['current_borrowers', 'title'],
        ascending=[False, True]
    )
    return qualified[['book_id', 'title', 'author', 'genre', 'publication_year', 'current_borrowers']]

# ----------------------------------------------------------------------
# Helper to create a borrowing DataFrame (ensures columns even if empty)
def make_borrowing_df(records):
    if records:
        return pd.DataFrame(records)
    else:
        return pd.DataFrame(columns=['record_id', 'book_id', 'borrower_name', 'borrow_date', 'return_date'])

In [27]:
# ----------------------------------------------------------------------
# Test cases
test_cases = []

# ----------------------------------------------------------------------
# 1–15: Specific small cases

# 1: Basic qualify
test_cases.append({
    'name': '01_basic_qualify',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 1
    }]),
    'borrowing': make_borrowing_df([{
        'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None
    }])
})

# 2: Basic not qualify
test_cases.append({
    'name': '02_basic_not_qualify',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 2
    }]),
    'borrowing': make_borrowing_df([{
        'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None
    }])
})

# 3: Over‑borrowed
test_cases.append({
    'name': '03_over_borrowed',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 1
    }]),
    'borrowing': make_borrowing_df([
        {'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None},
        {'record_id': 2, 'book_id': 1, 'borrower_name': 'B2', 'borrow_date': '2024-01-02', 'return_date': None}
    ])
})

# 4: Two books, one qualifies
test_cases.append({
    'name': '04_two_books_one_qualifies',
    'library': pd.DataFrame([
        {'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction', 'publication_year': 2000, 'total_copies': 1},
        {'book_id': 2, 'title': 'Book B', 'author': 'Author A', 'genre': 'Fiction', 'publication_year': 2000, 'total_copies': 2}
    ]),
    'borrowing': make_borrowing_df([
        {'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None},
        {'record_id': 2, 'book_id': 2, 'borrower_name': 'B2', 'borrow_date': '2024-01-01', 'return_date': None}
    ])
})

# 5: Two books, both qualify, different counts
test_cases.append({
    'name': '05_two_books_both_qualify',
    'library': pd.DataFrame([
        {'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction', 'publication_year': 2000, 'total_copies': 2},
        {'book_id': 2, 'title': 'Book B', 'author': 'Author A', 'genre': 'Fiction', 'publication_year': 2000, 'total_copies': 1}
    ]),
    'borrowing': make_borrowing_df([
        {'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None},
        {'record_id': 2, 'book_id': 1, 'borrower_name': 'B2', 'borrow_date': '2024-01-02', 'return_date': None},
        {'record_id': 3, 'book_id': 2, 'borrower_name': 'B3', 'borrow_date': '2024-01-01', 'return_date': None}
    ])
})

# 6: Zero copies, no borrows
test_cases.append({
    'name': '06_zero_copies_no_borrows',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 0
    }]),
    'borrowing': make_borrowing_df([])   # empty with columns
})

# 7: Zero copies, one borrow
test_cases.append({
    'name': '07_zero_copies_one_borrow',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 0
    }]),
    'borrowing': make_borrowing_df([{
        'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None
    }])
})

# 8: Mixed return status
test_cases.append({
    'name': '08_mixed_return_status',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 2
    }]),
    'borrowing': make_borrowing_df([
        {'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': '2024-01-10'},
        {'record_id': 2, 'book_id': 1, 'borrower_name': 'B2', 'borrow_date': '2024-01-02', 'return_date': None},
        {'record_id': 3, 'book_id': 1, 'borrower_name': 'B3', 'borrow_date': '2024-01-03', 'return_date': None}
    ])
})

# 9: No borrow records
test_cases.append({
    'name': '09_no_borrow_records',
    'library': pd.DataFrame([
        {'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction', 'publication_year': 2000, 'total_copies': 0},
        {'book_id': 2, 'title': 'Book B', 'author': 'Author A', 'genre': 'Fiction', 'publication_year': 2000, 'total_copies': 1}
    ]),
    'borrowing': make_borrowing_df([])   # empty with columns
})

# 10: Ordering same count
test_cases.append({
    'name': '10_ordering_same_count',
    'library': pd.DataFrame([
        {'book_id': 2, 'title': 'Z Book', 'author': 'Author A', 'genre': 'Fiction', 'publication_year': 2000, 'total_copies': 1},
        {'book_id': 1, 'title': 'A Book', 'author': 'Author A', 'genre': 'Fiction', 'publication_year': 2000, 'total_copies': 1}
    ]),
    'borrowing': make_borrowing_df([
        {'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None},
        {'record_id': 2, 'book_id': 2, 'borrower_name': 'B2', 'borrow_date': '2024-01-01', 'return_date': None}
    ])
})

# 11: Empty library
test_cases.append({
    'name': '11_empty_library',
    'library': pd.DataFrame(columns=['book_id', 'title', 'author', 'genre', 'publication_year', 'total_copies']),
    'borrowing': make_borrowing_df([
        {'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None}
    ])
})

# 12: Empty borrowing
test_cases.append({
    'name': '12_empty_borrowing',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 1
    }]),
    'borrowing': make_borrowing_df([])   # empty with columns
})

# 13: Multiple borrowers, all returned
test_cases.append({
    'name': '13_all_returned',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 2
    }]),
    'borrowing': make_borrowing_df([
        {'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': '2024-01-10'},
        {'record_id': 2, 'book_id': 1, 'borrower_name': 'B2', 'borrow_date': '2024-01-02', 'return_date': '2024-01-15'}
    ])
})

# 14: Zero copies, multiple borrowers
test_cases.append({
    'name': '14_zero_copies_multiple_borrowers',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': 0
    }]),
    'borrowing': make_borrowing_df([
        {'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None},
        {'record_id': 2, 'book_id': 1, 'borrower_name': 'B2', 'borrow_date': '2024-01-02', 'return_date': None}
    ])
})

# 15: Negative copies (invalid but test robustness)
test_cases.append({
    'name': '15_negative_copies',
    'library': pd.DataFrame([{
        'book_id': 1, 'title': 'Book A', 'author': 'Author A', 'genre': 'Fiction',
        'publication_year': 2000, 'total_copies': -1
    }]),
    'borrowing': make_borrowing_df([{
        'record_id': 1, 'book_id': 1, 'borrower_name': 'B1', 'borrow_date': '2024-01-01', 'return_date': None
    }])
})

# ----------------------------------------------------------------------
# 16–30: Single book with copies = current_borrowers for values 0..15
for copies in range(0, 16):
    test_name = f'16_{copies}_copies_{copies}_borrowers'
    library = pd.DataFrame([{
        'book_id': 1, 'title': 'Book', 'author': 'Author', 'genre': 'Genre',
        'publication_year': 2000, 'total_copies': copies
    }])
    records = [
        {'record_id': i+1, 'book_id': 1, 'borrower_name': f'B{i+1}', 'borrow_date': '2024-01-01', 'return_date': None}
        for i in range(copies)
    ]
    test_cases.append({
        'name': test_name,
        'library': library,
        'borrowing': make_borrowing_df(records)
    })

# ----------------------------------------------------------------------
# 31–60: Two‑book combinations
np.random.seed(42)
for idx in range(31, 61):
    # Randomly decide copies for book1 and book2 (0..5)
    copies1 = np.random.randint(0, 6)
    copies2 = np.random.randint(0, 6)
    # Randomly decide number of current borrowers for each (0..copies or could exceed)
    current1 = np.random.randint(0, copies1 + 2)
    current2 = np.random.randint(0, copies2 + 2)
    library = pd.DataFrame([
        {'book_id': 1, 'title': 'Apple', 'author': 'Author', 'genre': 'Genre', 'publication_year': 2000, 'total_copies': copies1},
        {'book_id': 2, 'title': 'Banana', 'author': 'Author', 'genre': 'Genre', 'publication_year': 2000, 'total_copies': copies2}
    ])
    records = []
    rec_id = 1
    for i in range(current1):
        records.append({
            'record_id': rec_id, 'book_id': 1, 'borrower_name': f'B1_{i}', 'borrow_date': '2024-01-01', 'return_date': None
        })
        rec_id += 1
    for i in range(current2):
        records.append({
            'record_id': rec_id, 'book_id': 2, 'borrower_name': f'B2_{i}', 'borrow_date': '2024-01-01', 'return_date': None
        })
        rec_id += 1
    test_cases.append({
        'name': f'31_60_two_books_{idx}',
        'library': library,
        'borrowing': make_borrowing_df(records)
    })

# ----------------------------------------------------------------------
# 61–80: Small random libraries (3–10 books, up to 30 borrow records)
for idx in range(61, 81):
    n_books = np.random.randint(3, 11)
    books = []
    for i in range(1, n_books+1):
        total_copies = np.random.randint(0, 8)
        books.append({
            'book_id': i,
            'title': f'Book_{i}',
            'author': f'Author_{i % 5}',
            'genre': f'Genre_{i % 3}',
            'publication_year': 1900 + np.random.randint(0, 121),
            'total_copies': total_copies
        })
    n_records = np.random.randint(0, n_books * 3)
    records = []
    for j in range(1, n_records+1):
        book_id = np.random.randint(1, n_books+1)
        return_date = None if np.random.random() < 0.7 else f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,28):02d}'
        records.append({
            'record_id': j,
            'book_id': book_id,
            'borrower_name': f'Borrower_{np.random.randint(1,100)}',
            'borrow_date': f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,28):02d}',
            'return_date': return_date
        })
    test_cases.append({
        'name': f'61_80_small_random_{idx}',
        'library': pd.DataFrame(books),
        'borrowing': make_borrowing_df(records)
    })

# ----------------------------------------------------------------------
# 81–100: Larger libraries (20–100 books, up to 200 records)
for idx in range(81, 101):
    n_books = np.random.randint(20, 101)
    books = []
    for i in range(1, n_books+1):
        total_copies = np.random.randint(0, 15)
        books.append({
            'book_id': i,
            'title': f'Book_{i}',
            'author': f'Author_{i % 10}',
            'genre': f'Genre_{i % 5}',
            'publication_year': 1900 + np.random.randint(0, 121),
            'total_copies': total_copies
        })
    n_records = np.random.randint(0, min(500, n_books * 5))
    records = []
    for j in range(1, n_records+1):
        book_id = np.random.randint(1, n_books+1)
        return_date = None if np.random.random() < 0.6 else f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,28):02d}'
        records.append({
            'record_id': j,
            'book_id': book_id,
            'borrower_name': f'Borrower_{np.random.randint(1,1000)}',
            'borrow_date': f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,28):02d}',
            'return_date': return_date
        })
    test_cases.append({
        'name': f'81_100_large_random_{idx}',
        'library': pd.DataFrame(books),
        'borrowing': make_borrowing_df(records)
    })

In [65]:
test_cases

[{'name': '01_basic_qualify',
  'library':    book_id   title    author    genre  publication_year  total_copies
  0        1  Book A  Author A  Fiction              2000             1,
  'borrowing':    record_id  book_id borrower_name borrow_date return_date
  0          1        1            B1  2024-01-01        None},
 {'name': '02_basic_not_qualify',
  'library':    book_id   title    author    genre  publication_year  total_copies
  0        1  Book A  Author A  Fiction              2000             2,
  'borrowing':    record_id  book_id borrower_name borrow_date return_date
  0          1        1            B1  2024-01-01        None},
 {'name': '03_over_borrowed',
  'library':    book_id   title    author    genre  publication_year  total_copies
  0        1  Book A  Author A  Fiction              2000             1,
  'borrowing':    record_id  book_id borrower_name borrow_date return_date
  0          1        1            B1  2024-01-01        None
  1          2        1

In [66]:
# ----------------------------------------------------------------------
# Now run all tests
def run_tests():
    print(f"Running {len(test_cases)} test cases...\n")
    passed = 0
    failed = 0
    for i, tc in enumerate(test_cases, 1):
        status = f"\rTest {i}....."
        # Compute expected using independent function
        expected = compute_expected(tc['library'], tc['borrowing'])
        expected.to_csv(f'test_{i}.csv')
        expected = pd.read_csv(f'test_{i}.csv', index_col=0)
        # Get solution result
        result = solution(tc['library'], tc['borrowing'])
        # Compare
        try:
            assert_frame_equal(result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False)
            print(status+"PASS", end='')
            passed += 1
        except AssertionError as e:
            print(status+"FAIL")
            failed += 1
            print("  Expected:")
            print(expected)
            print("  Got:")
            print(result)
            if len(expected) != len(result):
                print(f"  Lengths: expected {len(expected)}, got {len(result)}")
            break
    else:
        print('\nAll tests succesfull!')
    print(f"\nSummary: {passed} passed, {failed} failed out of {len(test_cases)} tests.")
run_tests()

Running 101 test cases...

Test 101.....PASS
All tests succesfull!

Summary: 101 passed, 0 failed out of 101 tests.
